In [1]:
# -*- coding: utf-8 -*-

# Sample Python code for youtube.channels.list
# See instructions for running these code samples locally:
# https://developers.google.com/explorer-help/code-samples#python

import os

import google_auth_oauthlib.flow
import googleapiclient.discovery
import googleapiclient.errors
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
import google.auth.exceptions
from google.auth.transport.requests import Request

import pickle
from convenient_pickle import *
import time
import datetime
import re

import country_converter as coco
from airports import airport_data
import pycountry

import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#You'll need to change your data path to get this to work
data_path = '[your_chosen_data_path]'
current_directory = os.getcwd()
scopes = ["https://www.googleapis.com/auth/youtube.force-ssl"]

In [3]:
def handle_credentials():
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
    quota = 0
    api_service_name = "youtube"
    api_version = "v3"
    # Change the below to whatever your secrets file is called.
    client_secrets_file = "[your_secrets_file]"
    credentials = None
    if os.path.exists('token.json'):
        try:
            credentials = Credentials.from_authorized_user_file('token.json',scopes)
            credentials.refresh(Request())
        except google.auth.exceptions.RefreshError as error: 
            credentials = None
            print(f'{error}')
    if not credentials or not credentials.valid: 
        if credentials and credentials.expired and credentials.refresh_token: 
            credentials.refresh(Request())
        else: 
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)
    with open('token.json', 'w') as token: 
        token.write(credentials.to_json())
    youtube = googleapiclient.discovery.build(
        api_service_name, api_version, credentials=credentials
        )
    return youtube

In [4]:
def get_video_urls(inlist, verbose=False): 
    outlist = []
    for search_page in inlist:
        #print(search_page)
        search_items = search_page['items']
        for item in search_items: 
            if 'videoId' in item['id'].keys():
                outlist.append(item['id']['videoId'])
            else: 
                print(item['id'])
    return outlist


def scrape_search_dict(search_list): 
    youtube = handle_credentials()
    info_list = []
    for url in search_list: 
        try:
            video_request = youtube.videos().list(
                part = "snippet,contentDetails,statistics",
                id = url
            ) 
            video_result = video_request.execute()
            info_list.append(video_result)
        except: 
            print("SOMETHING WENT WRONG, STOP THE SCRIPT")
            pass
    return info_list



In [5]:
total_len = 0
use_dict = dict()
len_dict=dict()
#german_country_names = ['Austria','Belgium','Switzerland','Germany']
albania_country_names = ['Albania']
#proper_videos = sorted([i for i in os.listdir(os.getcwd() + '/country_pickle_files') if 'youtube_travel_top_100' in i])
proper_videos = sorted([os.getcwd()+f'/country_pickle_files/youtube_travel_top_100_{i}.pkl'  for i in albania_country_names])
for file in proper_videos: 
    country_name = file.split('_')[-1].split('.')[0]
    test_video = load_pickle(file)
    check = list(set(get_video_urls(test_video)))
    total_len += len(check)
    use_dict[country_name] = check
    len_dict[country_name] = len(check)

Check the number of results in the country_results you've got. Bear in mind that you can only do 10 thousand videos per day.

In [6]:
os.getcwd()

'C:\\Users\\samue\\Documents\\OMSA class homework\\PRACTICUM\\Team 4 Code\\scraping\\Youtube Scrape'

In [10]:
print(len(use_dict['Albania']))
print(len(list(set(use_dict['Albania']))))

660
660


In [11]:
#place a start and end of where the countries you want to scrape are.
start = 0
end = len(use_dict.keys())

youtube = handle_credentials()

for country in list(use_dict.keys()): 
    scrape_urls = use_dict[country]
    country_videos = scrape_search_dict(scrape_urls)
    dump_pickle(os.getcwd(),f'/country_pickle_files/video_descriptions/video_descriptions_for_{country}.pkl', country_videos)

In [42]:
use_dict.keys()

dict_keys(['Albania'])